# 02 — Random Forest on synthetic data (Rust)
**Goal:** build a real random forest classifier using **only Rust's standard library**, without datasets, API keys or model downloads.

We'll generate a nonlinear, two-feature classification problem, train **21 bootstrapped decision trees**, use a randomly selected feature at each split, then evaluate accuracy and a confusion matrix on unseen data.

This is an educational implementation (binary classification, two numerical features), not a production ML framework. Run the cells **in order** using the **Rust** kernel.

In [1]:
#[derive(Clone, Copy)]
struct Sample { x: [f64; 2], label: u8 }

struct Rng { state: u64 }
impl Rng {
    fn new(seed: u64) -> Self { Self { state: seed.max(1) } }
    fn next(&mut self) -> u64 {
        self.state ^= self.state >> 12;
        self.state ^= self.state << 25;
        self.state ^= self.state >> 27;
        self.state.wrapping_mul(2685821657736338717)
    }
    fn unit(&mut self) -> f64 { ((self.next() >> 11) as f64) / ((1_u64 << 53) as f64) }
    fn index(&mut self, count: usize) -> usize { (self.next() % count as u64) as usize }
}
fn synthetic_data(seed: u64, count: usize) -> Vec<Sample> {
    let mut rng = Rng::new(seed);
    (0..count).map(|_| {
        let x = rng.unit() * 2.0 - 1.0;
        let y = rng.unit() * 2.0 - 1.0;
        let in_circle = x*x + y*y < 0.50;
        let in_corner = x > 0.35 && y < -0.35;
        Sample { x: [x, y], label: (in_circle ^ in_corner) as u8 }
    }).collect()
}
let train = synthetic_data(2026, 320);
let test = synthetic_data(2050, 120);
println!("Training: {} | Test: {}", train.len(), test.len());
println!("Training positives: {}", train.iter().filter(|p| p.label == 1).count());

Training: 320 | Test: 120


Training positives: 145


## 1. Decision trees

Each tree recursively chooses a numerical threshold that reduces **Gini impurity**. A forest adds variety with **bootstrap samples** (sampled with replacement) and **random feature selection** at each node.

In [2]:
enum Tree {
    Leaf(u8),
    Split { feature: usize, threshold: f64, left: Box<Tree>, right: Box<Tree> },
}
fn gini(ones: usize, total: usize) -> f64 {
    if total == 0 { return 0.0; }
    let p = ones as f64 / total as f64;
    2.0 * p * (1.0 - p)
}
fn grow(data: &[Sample], ids: &[usize], depth: usize, rng: &mut Rng) -> Tree {
    let positive = ids.iter().filter(|&&i| data[i].label == 1).count();
    let majority = (positive * 2 >= ids.len()) as u8;
    if depth == 0 || ids.len() < 6 || positive == 0 || positive == ids.len() {
        return Tree::Leaf(majority);
    }
    // Choose 1 of the 2 features at random at every split (mtry = 1).
    let feature = rng.index(2);
    let low = ids.iter().map(|&i| data[i].x[feature]).fold(f64::INFINITY, f64::min);
    let high = ids.iter().map(|&i| data[i].x[feature]).fold(f64::NEG_INFINITY, f64::max);
    if low == high { return Tree::Leaf(majority); }

    let parent_score = gini(positive, ids.len());
    let mut best_score = parent_score;
    let mut best_threshold = None;
    // Evaluate 15 candidate thresholds between the min and max feature values.
    for step in 1..=15 {
        let threshold = low + (high - low) * (step as f64) / 16.0;
        let mut left_n = 0;
        let mut left_ones = 0;
        for &id in ids {
            if data[id].x[feature] <= threshold {
                left_n += 1;
                left_ones += data[id].label as usize;
            }
        }
        if left_n == 0 || left_n == ids.len() { continue; }
        let right_n = ids.len() - left_n;
        let right_ones = positive - left_ones;
        let score = (left_n as f64 * gini(left_ones, left_n)
            + right_n as f64 * gini(right_ones, right_n)) / ids.len() as f64;
        if score < best_score {
            best_score = score;
            best_threshold = Some(threshold);
        }
    }
    match best_threshold {
        None => Tree::Leaf(majority),
        Some(threshold) => {
            let (left_ids, right_ids): (Vec<usize>, Vec<usize>) = ids.iter()
                .copied().partition(|&i| data[i].x[feature] <= threshold);
            Tree::Split {
                feature, threshold,
                left: Box::new(grow(data, &left_ids, depth - 1, rng)),
                right: Box::new(grow(data, &right_ids, depth - 1, rng)),
            }
        }
    }
}
fn classify(tree: &Tree, features: [f64; 2]) -> u8 {
    match tree {
        Tree::Leaf(value) => *value,
        Tree::Split { feature, threshold, left, right } => {
            if features[*feature] <= *threshold { classify(left, features) }
            else { classify(right, features) }
        }
    }
}

## 2. Train the Random Forest

Each tree trains on its own random bootstrap sample. At prediction time, the **majority vote** of all trees is the forest's prediction.

In [3]:
fn fit_forest(data: &[Sample], trees: usize, depth: usize, seed: u64) -> Vec<Tree> {
    assert!(!data.is_empty() && trees > 0);
    let mut rng = Rng::new(seed);
    (0..trees).map(|_| {
        let bootstrap: Vec<usize> = (0..data.len()).map(|_| rng.index(data.len())).collect();
        grow(data, &bootstrap, depth, &mut rng)
    }).collect()
}
fn forest_predict(forest: &[Tree], features: [f64; 2]) -> u8 {
    let votes = forest.iter().filter(|tree| classify(tree, features) == 1).count();
    (votes * 2 > forest.len()) as u8
}
let forest = fit_forest(&train, 21, 8, 42);
println!("Trained {} randomized trees", forest.len());

Trained 21 randomized trees


## 3. Evaluate on new synthetic data

The test set uses a **different seed**, so the forest cannot memorize the exact training observations. Print the confusion matrix and accuracy, and compare with a majority-class baseline.

In [4]:
let mut confusion = [[0_usize; 2]; 2]; // [true label][predicted label]
for row in &test {
    let predicted = forest_predict(&forest, row.x);
    confusion[row.label as usize][predicted as usize] += 1;
}
let correct = confusion[0][0] + confusion[1][1];
let accuracy = correct as f64 / test.len() as f64;
let baseline = test.iter().filter(|r| r.label == 1).count();
let baseline_accuracy = baseline.max(test.len() - baseline) as f64 / test.len() as f64;
println!("Accuracy: {:.1}%", accuracy * 100.0);
println!("Majority baseline: {:.1}%", baseline_accuracy * 100.0);
println!("Confusion matrix [true][predicted]: {:?}", confusion);
assert!(accuracy > baseline_accuracy, "Forest should improve on the majority baseline");

Accuracy: 92.5%


Majority baseline: 50.8%


Confusion matrix [true][predicted]: [[57, 4], [5, 54]]


## 4. Your experiment

Change **the number of trees** (21 → 3 or 51) or **maximum depth** (8 → 3 or 12). Rerun the training and evaluation cells to see how your model changes. Because seeds are fixed, runs are reproducible.

**Next:** [03 — Neural network](03_neural_network.ipynb) to train a small nonlinear model with backpropagation.